# Flux LoRA Training — Colab T4

Тренировка кастомной LoRA конкретного человека с помощью **ai-toolkit** (Ostris) на бесплатном T4 GPU.

**Что вы получите:** LoRA файл (.safetensors), который можно использовать в ComfyUI с нодой LoraLoader для генерации изображений человека в любом стиле/окружении.

**Время:** ~30-60 мин тренировки на T4
**VRAM:** ~14 GB пиковое потребление

**Запускайте ячейки 1-5 по порядку.**

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
#@title 2. Установка ai-toolkit + Зависимости
%cd /content
!git clone https://github.com/ostris/ai-toolkit.git 2>/dev/null || echo "Уже склонировано"
%cd /content/ai-toolkit
!git submodule update --init --recursive
!pip install -r requirements.txt -q
!pip install peft accelerate safetensors PyYAML -q

print("\nai-toolkit установлен!")

In [ ]:
#@title 3. Загрузка тренировочных фото (15-30 шт)
#@markdown Загрузите 15-30 фотографий человека. Больше разнообразия = лучше результат.
#@markdown - Разные ракурсы (анфас, 3/4, профиль)
#@markdown - Разное освещение (в помещении, на улице, студийное)
#@markdown - Разные выражения лица
#@markdown - Чёткое лицо, хорошее качество

from google.colab import files
import os

DATASET_DIR = "/content/dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    path = os.path.join(DATASET_DIR, name)
    with open(path, "wb") as f:
        f.write(data)
    print(f"Сохранено: {path}")

count = len(os.listdir(DATASET_DIR))
print(f"\nВсего изображений: {count}")
if count < 10:
    print("ВНИМАНИЕ: Менее 10 изображений. Рекомендуется: 15-30 для лучшего результата.")

In [ ]:
#@title 4. Настройка и запуск тренировки
#@markdown Задайте параметры тренировки ниже.

trigger_word = "ohwx" #@param {type:"string"}
#@markdown Триггер-слово — уникальный токен, который активирует LoRA. Используйте что-то редкое, например "ohwx" или "sks".

steps = 1500 #@param {type:"integer"}
#@markdown Количество шагов тренировки (рекомендуется 1000-2000 для T4)

learning_rate = 1e-4 #@param {type:"number"}
resolution = 512 #@param [512, 768] {type:"raw"}
lora_rank = 16 #@param [8, 16, 32] {type:"raw"}
output_name = "my_person_lora" #@param {type:"string"}

import yaml, os

config = {
    "job": "extension",
    "config": {
        "name": output_name,
        "process": [{
            "type": "sd_trainer",
            "training_folder": "/content/output",
            "performance_log_every": 100,
            "device": "cuda:0",
            "trigger_word": trigger_word,
            "network": {
                "type": "lora",
                "linear": lora_rank,
                "linear_alpha": lora_rank
            },
            "save": {
                "dtype": "float16",
                "save_every": 500,
                "max_step_saves_to_keep": 2
            },
            "datasets": [{
                "folder_path": "/content/dataset",
                "caption_ext": "txt",
                "caption_dropout_rate": 0.05,
                "shuffle_tokens": False,
                "cache_latents_to_disk": True,
                "resolution": [resolution, resolution]
            }],
            "train": {
                "batch_size": 1,
                "steps": steps,
                "gradient_accumulation_steps": 1,
                "train_unet": True,
                "train_text_encoder": False,
                "gradient_checkpointing": True,
                "noise_scheduler": "flowmatch",
                "optimizer": "adamw8bit",
                "lr": learning_rate,
                "ema_config": {"use_ema": True, "ema_decay": 0.99},
                "dtype": "bf16"
            },
            "model": {
                "name_or_path": "black-forest-labs/FLUX.1-dev",
                "is_flux": True,
                "quantize": True
            },
            "sample": {
                "sampler": "flowmatch",
                "sample_every": 500,
                "width": resolution,
                "height": resolution,
                "prompts": [
                    f"a photo of {trigger_word} person, professional headshot, studio lighting",
                    f"a photo of {trigger_word} person, casual outdoor portrait, natural light"
                ],
                "neg": "",
                "seed": 42,
                "walk_seed": True,
                "guidance_scale": 3.5,
                "sample_steps": 25
            }
        }]
    }
}

config_path = "/content/ai-toolkit/config/train_lora.yaml"
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Конфиг сохранён: {config_path}")
print(f"\nПараметры тренировки:")
print(f"  Триггер-слово: {trigger_word}")
print(f"  Шаги: {steps}")
print(f"  Скорость обучения: {learning_rate}")
print(f"  Разрешение: {resolution}x{resolution}")
print(f"  LoRA rank: {lora_rank}")
print(f"\nЗапуск тренировки... (это займёт 30-60 мин на T4)")

!cd /content/ai-toolkit && python run.py config/train_lora.yaml

print("\nТренировка завершена!")

In [ ]:
#@title 5. Тест LoRA — Генерация тестового фото
#@markdown Быстрый тест обученной LoRA через пайплайн diffusers.

trigger_word = "ohwx" #@param {type:"string"}
prompt = "a photo of ohwx person, professional headshot, studio lighting, 4k" #@param {type:"string"}
output_name = "my_person_lora" #@param {type:"string"}

import glob

# Находим последний LoRA файл
lora_files = sorted(glob.glob(f"/content/output/{output_name}/*.safetensors"))
if not lora_files:
    print("LoRA файл не найден! Убедитесь, что тренировка завершена.")
else:
    lora_path = lora_files[-1]
    print(f"Используется LoRA: {lora_path}")

    import torch
    from diffusers import FluxPipeline

    pipe = FluxPipeline.from_pretrained(
        "black-forest-labs/FLUX.1-dev",
        torch_dtype=torch.bfloat16
    ).to("cuda")
    pipe.load_lora_weights(lora_path)

    image = pipe(
        prompt,
        num_inference_steps=25,
        guidance_scale=3.5,
        generator=torch.Generator("cuda").manual_seed(42)
    ).images[0]

    image.save("/content/test_result.png")

    from IPython.display import display, Image
    display(Image("/content/test_result.png"))
    print("Тестовое изображение сохранено: /content/test_result.png")

    del pipe
    import gc; gc.collect()
    torch.cuda.empty_cache()

In [ ]:
#@title 6. Скачивание LoRA / Сохранение на Google Drive
#@markdown Выберите, куда сохранить обученный LoRA файл.

save_to_drive = True #@param {type:"boolean"}
output_name = "my_person_lora" #@param {type:"string"}

import glob, shutil, os

lora_files = sorted(glob.glob(f"/content/output/{output_name}/*.safetensors"))
if not lora_files:
    print("LoRA файл не найден!")
else:
    lora_path = lora_files[-1]
    lora_filename = os.path.basename(lora_path)
    print(f"LoRA файл: {lora_path} ({os.path.getsize(lora_path) / 1024**2:.1f} MB)")

    if save_to_drive:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_dir = "/content/drive/MyDrive/ComfyUI_LoRAs"
        os.makedirs(drive_dir, exist_ok=True)
        dest = f"{drive_dir}/{lora_filename}"
        shutil.copy2(lora_path, dest)
        print(f"Сохранено на Drive: {dest}")
    else:
        from google.colab import files
        files.download(lora_path)
        print("Скачивание начато!")

---
## Использование LoRA в ComfyUI

### Способ 1: Нода LoraLoader
1. Скопируйте LoRA файл в `ComfyUI/models/loras/`
2. В вашем воркфлоу добавьте ноду **LoraLoader**
3. Подключите её между загрузчиком модели и KSampler
4. Выберите ваш LoRA файл
5. Используйте триггер-слово в промпте (например, "a photo of ohwx person in a garden")

### Способ 2: В Colab
Используйте ячейку "Скачать LoRA" в `colab_flux_photo.ipynb` для прямой загрузки.

### Советы
- **Триггер-слово** должно присутствовать в промпте для активации LoRA
- **Сила (strength)** 0.7-1.0 лучше всего работает для LoRA лиц
- **Пониженная сила** (0.4-0.6) для большего стилистического разнообразия
- Комбинируйте с IPAdapter FaceID для ещё лучшей консистентности

### Решение проблем
- **Переобучение (overfit)** (все изображения выглядят одинаково): уменьшите количество шагов, увеличьте разнообразие датасета
- **Недообучение (underfit)** (лицо не распознаётся): увеличьте количество шагов, улучшите качество изображений
- **Ошибки OOM**: уменьшите разрешение до 512, уменьшите batch_size до 1